In [ ]:
# Quantas ordens possuem cliente válido?

# Quantas ordens estão sem cliente cadastrado?

# Quais ordens possuem `id_cliente` sem correspondência na tabela de clientes?

# Qual foi o faturamento total válido considerando apenas ordens finalizadas com cliente cadastrado?

# Quanto cada cliente válido faturou?

In [1]:
# criando dataframes
import pandas as pd

clientes = pd.read_csv('../data/clientes.csv')
ordens = pd.read_csv('../data/ordens_servico.csv')

display(clientes, ordens)

,id_cliente,nome_cliente,cidade
0,1,João,São Paulo
1,2,Maria,São Paulo
2,3,Ana,Guarulhos
3,4,Carlos,Suzano
4,5,Pedro,Mogi das Cruzes
5,6,Juliana,Guarulhos


,id_os,id_cliente,servico,valor,status
0,1,1,Troca de Óleo,120,Finalizado
1,2,2,Alinhamento,80,Finalizado
2,3,1,Freio,350,Finalizado
3,4,4,Troca de Óleo,120,Pendente
4,5,2,Freio,300,Finalizado
5,6,3,Revisão,500,Finalizado
6,7,1,Revisão,500,Finalizado
7,8,4,Alinhamento,80,Finalizado
8,9,3,Freio,320,Pendente
9,10,2,Revisão,500,Finalizado


In [6]:
# unindo dataframes

df = clientes.merge(ordens, how='inner', on='id_cliente')

display(df)

,id_cliente,nome_cliente,cidade,id_os,servico,valor,status
0,1,João,São Paulo,1,Troca de Óleo,120,Finalizado
1,1,João,São Paulo,3,Freio,350,Finalizado
2,1,João,São Paulo,7,Revisão,500,Finalizado
3,2,Maria,São Paulo,2,Alinhamento,80,Finalizado
4,2,Maria,São Paulo,5,Freio,300,Finalizado
5,2,Maria,São Paulo,10,Revisão,500,Finalizado
6,3,Ana,Guarulhos,6,Revisão,500,Finalizado
7,3,Ana,Guarulhos,9,Freio,320,Pendente
8,3,Ana,Guarulhos,15,Alinhamento,90,Finalizado
9,4,Carlos,Suzano,4,Troca de Óleo,120,Pendente


In [9]:
# filtrando por status finalizado

df = df[df['status'] == 'Finalizado']
display(df)

,id_cliente,nome_cliente,cidade,id_os,servico,valor,status
0,1,João,São Paulo,1,Troca de Óleo,120,Finalizado
1,1,João,São Paulo,3,Freio,350,Finalizado
2,1,João,São Paulo,7,Revisão,500,Finalizado
3,2,Maria,São Paulo,2,Alinhamento,80,Finalizado
4,2,Maria,São Paulo,5,Freio,300,Finalizado
5,2,Maria,São Paulo,10,Revisão,500,Finalizado
6,3,Ana,Guarulhos,6,Revisão,500,Finalizado
8,3,Ana,Guarulhos,15,Alinhamento,90,Finalizado
10,4,Carlos,Suzano,8,Alinhamento,80,Finalizado
11,5,Pedro,Mogi das Cruzes,11,Bateria,450,Finalizado


In [11]:
# Quantas ordens possuem cliente válido?

ordens_cliente = df.groupby('nome_cliente')['id_os'].count()

In [ ]:
# Quantas ordens estão sem cliente cadastrado?

os_ausente = clientes.merge(ordens, how='right', on='id_cliente').fillna('Ausente')
os_ausente = os_ausente[os_ausente['nome_cliente'] == 'Ausente']

ordens_sem_cliente = os_ausente.shape[0]

display(os_ausente, ordens_sem_cliente)

,id_cliente,nome_cliente,cidade,id_os,servico,valor,status
15,99,Ausente,Ausente,16,Revisão,600,Finalizado
16,100,Ausente,Ausente,17,Freio,400,Finalizado


2

In [28]:
# Quais ordens possuem `id_cliente` sem correspondência na tabela de clientes?
os_cliente_ausente = os_ausente['id_os']

display(os_cliente_ausente)

15    16
16    17
Name: id_os, dtype: int64

In [ ]:
# Qual foi o faturamento total válido considerando apenas ordens finalizadas com cliente cadastrado?

faturamento_total = df['valor'].sum()

display(faturamento_total)

np.int64(3470)

In [30]:
# Quanto cada cliente válido faturou?
faturamento_cliente = df.groupby('nome_cliente')['valor'].sum()

display(faturamento_cliente)

nome_cliente
Ana        590
Carlos      80
João       970
Juliana    500
Maria      880
Pedro      450
Name: valor, dtype: int64

In [38]:
# criando relatorio por cliente
relatorio = pd.DataFrame({
    'id_cliente': df.groupby('nome_cliente')['id_cliente'].first(),
    'nome': faturamento_cliente.index,
    'cidade': df.groupby('nome_cliente')['cidade'].first(),
    'faturamento_total': df.groupby('nome_cliente')['valor'].sum(),
    'qtd_ordens': df.groupby('nome_cliente')['id_os'].count(),
    'ticket_medio': df.groupby('nome_cliente')['valor'].mean().round(1)
}).sort_values('id_cliente')

display(relatorio)



,id_cliente,nome,cidade,faturamento_total,qtd_ordens,ticket_medio
nome_cliente,,,,,,
João,1,João,São Paulo,970,3,323.3
Maria,2,Maria,São Paulo,880,3,293.3
Ana,3,Ana,Guarulhos,590,2,295.0
Carlos,4,Carlos,Suzano,80,1,80.0
Pedro,5,Pedro,Mogi das Cruzes,450,1,450.0
Juliana,6,Juliana,Guarulhos,500,2,250.0


In [39]:
# exportando relatorios
relatorio.to_csv('../output/relatorio.csv', index=False)

os_ausente.to_csv('../output/ordens_sem_cliente.csv', index=False)